Title - Breast Cancer Detection System using Simple Neural Networks. Use Tensorflow and keras to create a neural network to detect possibility of breast cancer

1. Import Necessary Libraries

In [1]:
!pip install tensorflow

Defaulting to user installation because normal site-packages is not writeable


In [3]:
import os
import json
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import random
SEED = 42
random.seed(SEED)

import numpy as np
np.random.seed(SEED)

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

import tensorflow as tf
tf.random.set_seed(SEED)
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

BASE_DIR    = os.path.dirname(os.path.abspath('__file__'))
DATA_PATH   = os.path.join(BASE_DIR, 'data',   'breast_cancer.csv')
MODEL_PATH  = os.path.join(BASE_DIR, 'models', 'saved_model.keras')
PLOTS_DIR   = os.path.join(BASE_DIR, 'outputs', 'plots')
METRICS_DIR = os.path.join(BASE_DIR, 'outputs', 'metrics')
PREDS_DIR   = os.path.join(BASE_DIR, 'outputs', 'predictions')

for d in [PLOTS_DIR, METRICS_DIR, PREDS_DIR,
          os.path.join(BASE_DIR, 'models')]:
    os.makedirs(d, exist_ok=True)

print('All libraries imported successfully!')
print(f'   NumPy      {np.__version__}')
print(f'   Pandas     {pd.__version__}')
print(f'   TensorFlow {tf.__version__}')
print(f'   Matplotlib {plt.matplotlib.__version__}')
print(f'\nOutput directories ready.')

All libraries imported successfully!
   NumPy      2.3.5
   Pandas     2.3.3
   TensorFlow 2.21.0
   Matplotlib 3.10.6

Output directories ready.


2. Loading the Dataset

In [ ]:
df = pd.read_csv(DATA_PATH)

print(f'Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Data path : {DATA_PATH}\n')
df.head()

Dataset loaded: 569 rows × 32 columns
Data path : c:\Users\Udbhav\Documents\ML-Projects\Breat-Cancer-Detection\data\breast_cancer.csv



,diagnosis,diagnosis_label,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,0,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,0,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,0,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,0,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


3. Exploratory Data Analysis

In [5]:
print(f'  Shape          : {df.shape}')
print(f'  Feature count  : {df.shape[1] - 2}  (excl. target columns)')
print(f'  Sample count   : {df.shape[0]}')
print(f'  Memory usage   : {df.memory_usage(deep=True).sum() / 1024:.1f} KB')
print('\n  Column Data Types:')
print(df.dtypes.value_counts().to_string())

  Shape          : (569, 32)
  Feature count  : 30  (excl. target columns)
  Sample count   : 569
  Memory usage   : 165.7 KB

  Column Data Types:
float64    30
int64       1
object      1


In [6]:
feature_cols = [c for c in df.columns if c not in ['diagnosis', 'diagnosis_label']]
df[feature_cols].describe().round(3)

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
count,569.000,569.000,569.000,569.000,569.000,569.000,569.000,569.000,569.000,569.000,...,569.000,569.000,569.000,569.000,569.000,569.000,569.000,569.000,569.000,569.000
mean,14.127,19.290,91.969,654.889,0.096,0.104,0.089,0.049,0.181,0.063,...,16.269,25.677,107.261,880.583,0.132,0.254,0.272,0.115,0.290,0.084
std,3.524,4.301,24.299,351.914,0.014,0.053,0.080,0.039,0.027,0.007,...,4.833,6.146,33.603,569.357,0.023,0.157,0.209,0.066,0.062,0.018
min,6.981,9.710,43.790,143.500,0.053,0.019,0.000,0.000,0.106,0.050,...,7.930,12.020,50.410,185.200,0.071,0.027,0.000,0.000,0.156,0.055
25%,11.700,16.170,75.170,420.300,0.086,0.065,0.030,0.020,0.162,0.058,...,13.010,21.080,84.110,515.300,0.117,0.147,0.114,0.065,0.250,0.071
50%,13.370,18.840,86.240,551.100,0.096,0.093,0.062,0.034,0.179,0.062,...,14.970,25.410,97.660,686.500,0.131,0.212,0.227,0.100,0.282,0.080
75%,15.780,21.800,104.100,782.700,0.105,0.130,0.131,0.074,0.196,0.066,...,18.790,29.720,125.400,1084.000,0.146,0.339,0.383,0.161,0.318,0.092
max,28.110,39.280,188.500,2501.000,0.163,0.345,0.427,0.201,0.304,0.097,...,36.040,49.540,251.200,4254.000,0.223,1.058,1.252,0.291,0.664,0.208


In [7]:
#Check for missing values

missing = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if missing_df.empty:
    print('No missing values found! Dataset is clean.')
else:
    print(f'Missing values detected:')
    print(missing_df)

No missing values found! Dataset is clean.


In [9]:
class_counts = df['diagnosis_label'].value_counts()
class_pct    = df['diagnosis_label'].value_counts(normalize=True).mul(100).round(2)

print('Class Distribution:')
print('─' * 35)
for label, count, pct in zip(class_counts.index, class_counts.values, class_pct.values):
    name = 'Benign (B)   ' if label == 'B' else 'Malignant (M)'
    bar  = '[]' * int(pct / 2)
    print(f'  {name}  {count:>4}  ({pct:>5.1f}%)  {bar}')
print('─' * 35)
print(f'  Total         :  {len(df)}')
imbalance_ratio = class_counts.max() / class_counts.min()
print(f'  Imbalance ratio: {imbalance_ratio:.2f} : 1')

Class Distribution:
───────────────────────────────────
  Benign (B)      357  ( 62.7%)  [][][][][][][][][][][][][][][][][][][][][][][][][][][][][][][]
  Malignant (M)   212  ( 37.3%)  [][][][][][][][][][][][][][][][][][]
───────────────────────────────────
  Total         :  569
  Imbalance ratio: 1.68 : 1


4. Data Visualization

In [11]:
def save_plot(filename, dpi=150):
    """Save current figure to outputs/plots/ directory."""
    path = os.path.join(PLOTS_DIR, filename)
    plt.savefig(path, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'   💾 Saved → {path}')

COLORS = {'B': '#2196F3', 'M': '#F44336'}   # Blue = Benign, Red = Malignant
PALETTE = ['#2196F3', '#F44336']

print('Colour palette defined. Starting visualizations…')

Colour palette defined. Starting visualizations…
